# masaMLP — CUDA verification (Colab)

Runtime > Change runtime type > **GPU** (T4 is fine).

While the repo is private, add a GitHub token once via the key icon
(Colab **Secrets**) as `GH_TOKEN` with repo-read access, and enable
notebook access for it. Once the repo is public, replace the clone cell
with a plain `!git clone https://github.com/Matapanino/masamlp.git`.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git clone https://{token}@github.com/Matapanino/masamlp.git
%pip install -q -e ./masamlp[dev]

## Full test suite on a CUDA machine

`tests/test_device.py::test_cpu_cuda_parity` un-skips itself here; the
rest of the suite verifies everything still behaves with CUDA visible.

In [ ]:
!cd masamlp && python -m pytest tests/ -q

## GPU-only smoke: every model trains on CUDA

In [ ]:
import numpy as np
import sys; sys.path.insert(0, 'masamlp/src')
from masamlp import MasaClassifier
from masamlp.models import _MODEL_REGISTRY

rng = np.random.default_rng(0)
X = rng.normal(size=(4000, 10)).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(int)
for name in ['resnet', 'realmlp', 'ft_transformer', 'tab_transformer',
             'danet', 'tabr', 'modernnca', 'gandalf', 'grn', 'lnn']:
    clf = MasaClassifier(model=name, n_epochs=10, device='cuda', random_state=0)
    clf.fit(X[:3000], y[:3000])
    acc = (clf.predict(X[3000:]) == y[3000:]).mean()
    print(f'{name:16s} cuda acc={acc:.3f}')

## Speed benchmark (CPU vs CUDA, AMP, vectorized ensembles)

In [ ]:
!cd masamlp && python benchmarks/gpu_speed.py --rows 50000